In [2]:
%pip install requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.2-cp314-cp314-win_amd64.whl.metadata (47 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)

   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ------------- -------------------------- 1/3 [charset_normalizer]
   ------------- -------------------------- 1/3 [charset_normalizer]
   ------------- -------------------------- 1/3 [charset_normalizer]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import requests

In [5]:
url = "https://api.github.com/search/repositories"

params = {
    "q": "language:python created:>2023-01-01",
    "sort": "stars",
    "order": "desc",
    "per_page": 30
}

response = requests.get(url, params=params)
data = response.json()

print(response.status_code)
print(data.keys())

200
dict_keys(['total_count', 'incomplete_results', 'items'])


In [6]:
print(f"Total repositories found: {data['total_count']}")
print("Repos retrieved in this request:", len(data['items']))

first_repo = data['items'][0]
print(f"First repository name: {first_repo['name']}")
print(f"First repository stars: {first_repo['stargazers_count']}")
print(f"First repository URL: {first_repo['html_url']}")
print("Language of the first repository:", first_repo['language'])

Total repositories found: 24317069
Repos retrieved in this request: 30
First repository name: hermes-agent
First repository stars: 265230
First repository URL: https://github.com/NousResearch/hermes-agent
Language of the first repository: Python


In [7]:
import pandas as pd

In [8]:
repos = []

for repo in data['items']:
    repos.append({
        "name": repo['name'],
        "owner": repo['owner']['login'],
        "stars": repo['stargazers_count'],
        "forks": repo['forks_count'],
        "language": repo['language'],
        "description": repo['description'],
        "url": repo['html_url'],
        "created_at": repo['created_at']
    })

df = pd.DataFrame(repos)
df.head(10)

,name,owner,stars,forks,language,description,url,created_at
0,hermes-agent,NousResearch,265230,59956,Python,The agent that grows with you,https://github.com/NousResearch/hermes-agent,2025-07-22T22:22:28Z
1,skills,anthropics,198569,23239,Python,Public repository for Agent Skills,https://github.com/anthropics/skills,2025-09-22T15:53:31Z
2,AutoGPT,Significant-Gravitas,188124,45827,Python,AutoGPT is the vision of accessible AI for eve...,https://github.com/Significant-Gravitas/AutoGPT,2023-03-16T09:21:07Z
3,ui-ux-pro-max-skill,nextlevelbuilder,152563,16084,Python,An AI skill that provides design intelligence ...,https://github.com/nextlevelbuilder/ui-ux-pro-...,2025-11-30T11:36:31Z
4,spec-kit,github,150956,13558,Python,💫 Toolkit to help you get started with SDD or ...,https://github.com/github/spec-kit,2025-08-21T22:54:31Z
5,ComfyUI,Comfy-Org,148001,17625,Python,The most powerful and modular diffusion model ...,https://github.com/Comfy-Org/ComfyUI,2023-01-17T03:15:56Z
6,DeepSeek-V3,deepseek-ai,104891,16737,Python,NaN,https://github.com/deepseek-ai/DeepSeek-V3,2024-12-26T09:52:40Z
7,Scrapling,D4Vinci,104860,10969,Python,🕷️ The BEST Web Scraping framework that handle...,https://github.com/D4Vinci/Scrapling,2024-10-13T20:29:53Z
8,autoresearch,karpathy,100391,13792,Python,AI agents running research on single-GPU nanoc...,https://github.com/karpathy/autoresearch,2026-03-06T22:00:43Z
9,odysseus,odysseus-dev,95896,2017,Python,Self-hosted AI workspace.,https://github.com/odysseus-dev/odysseus,2026-05-31T14:05:51Z


In [9]:
df_clean = df.dropna(subset=['description'])

df_clean = df_clean.copy()
df_clean['viral'] = df_clean['stars'].apply(lambda x: 'Yes' if x > 1000 else 'No')

df_clean = df_clean.sort_values(by='stars', ascending=False).reset_index(drop=True)

print("Before cleaning:", len(df))
print("After cleaning:", len(df_clean))

Before cleaning: 30
After cleaning: 29


In [10]:
df_clean.to_csv('github_trending_repos.csv', index=False)

print("Data saved to 'github_trending_repos.csv'")
print(f"Repos loaded into github_trending_repos.csv: {len(df_clean)}")

Data saved to 'github_trending_repos.csv'
Repos loaded into github_trending_repos.csv: 29


In [11]:
import sqlite3

In [ ]:
conn = sqlite3.connect('github_trending_repos.db')
cur = conn.cursor()

cur.execute("""CREATE TABLE IF NOT EXISTS repos (
    name TEXT,
    owner TEXT,
    stars INTEGER,
    forks INTEGER,
    language TEXT,
    description TEXT,
    url TEXT,
    created_at TEXT
    PRIMARY KEY (owner, url)
   )""")
conn.commit()

In [15]:
rows = df_clean[
    ["name", "owner", "stars", "forks", "language", "description", "url", "created_at"]
].values.tolist()

cur.executemany(
    """INSERT OR REPLACE INTO repos (name, owner, stars, forks, language, description, url, created_at) VALUES (?, ?, ?, ?, ?, ?, ? ,?)""", rows
)

conn.commit()
print(f"Pipeline complete. {len(rows)} rows laoded into github_trending_repos.db")

Pipeline complete. 29 rows laoded into github_trending_repos.db


In [16]:
print(pd.read_sql("SELECT COUNT(*) AS total_rows FROM repos", conn))

query = """
    SELECT name,
           ROUND(AVG(stars), 1)   AS avg_high,
           ROUND(AVG(forks), 1)   AS avg_low,
           ROUND(SUM(language), 1) AS total_language
    FROM repos
    GROUP BY name
    ORDER BY total_language DESC
"""
print(pd.read_sql(query, conn))

   total_rows
0          29
                    name  avg_high  avg_low  total_language
0    ui-ux-pro-max-skill  152563.0  16084.0             0.0
1   system_prompts_leaks   72344.0  11643.0             0.0
2               spec-kit  150956.0  13558.0             0.0
3                 skills  198569.0  23239.0             0.0
4     screenshot-to-code   81342.0   9956.0             0.0
5             ppt-master   79103.0   6163.0             0.0
6               odysseus   95896.0   2017.0             0.0
7                litellm   63674.0  13467.0             0.0
8      learn-claude-code   80706.0  12877.0             0.0
9       last30days-skill   72117.0   6386.0             0.0
10          hermes-agent  265230.0  59956.0             0.0
11          hello-agents   87586.0  10940.0             0.0
12              headroom   80266.0   6284.0             0.0
13          gpt_academic   71706.0   8289.0             0.0
14              gpt4free   67098.0  13436.0             0.0
15      free